In [1]:
import sys, platform, importlib.metadata, json
print("AUDIT_KERNEL="+json.dumps({"python":sys.version,"executable":sys.executable,"platform":platform.platform(),"packages":dict(sorted((d.metadata["Name"],d.version) for d in importlib.metadata.distributions()))}))

AUDIT_KERNEL={"python": "3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]", "executable": "C:\\Users\\Miakhlov\\Documents\\GitHub\\Computational chemistry_course\\scripts\\environment\\.venv-chem\\Scripts\\python.exe", "platform": "Windows-10-10.0.19045-SP0", "packages": {"Jinja2": "3.1.6", "MarkupSafe": "3.0.4", "PyYAML": "6.0.3", "Pygments": "2.21.0", "Send2Trash": "2.2.0", "altair": "6.3.0", "anyio": "4.16.0", "argon2-cffi": "25.1.0", "argon2-cffi-bindings": "26.2.0", "arrow": "1.4.0", "asttokens": "3.0.2", "async-lru": "2.4.0", "attrs": "26.1.0", "babel": "2.18.0", "beautifulsoup4": "4.15.0", "bleach": "6.4.0", "certifi": "2026.10.23", "cffi": "2.1.1", "charset-normalizer": "3.5.2", "click": "8.5.1", "cloudpickle": "3.1.3", "colorama": "0.4.6", "comm": "0.2.3", "contourpy": "1.4.0", "cycler": "0.12.1", "debugpy": "1.8.22", "defusedxml": "0.7.1", "executing": "2.3.0", "fastjsonschema": "2.22.3", "fonttools": "4.68.0", "fqdn": "1.6.0", "h11": "0.16.0

# Лабораторная B. В досье четыре формы — какое сравнение допустимо?

Аналитик получил записи «аторвастатин», «аторвастатин кальция» и «лактон аторвастатина». Неизвестно, какому именно материалу соответствует будущая проба. Подготовьте реестр так, чтобы следующий модуль мог рассчитывать свойства без молчаливой подмены объекта.

In [2]:
from pathlib import Path
import sys
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import compound,workspace,init_dossier,submission_feedback
from rdkit import Chem
from rdkit.Chem import Descriptors,rdMolDescriptors
OUT=workspace('M03-L02','lab')
DOSSIER=init_dossier()
records=[compound(name) for name in ['atorvastatin','atorvastatin_calcium','atorvastatin_lactone']]
display(records)


[{'name': 'atorvastatin',
  'cid': 60823,
  'smiles': 'CC(C)C1=C(C(=C(N1CC[C@H](C[C@H](CC(=O)O)O)O)C2=CC=C(C=C2)F)C3=CC=CC=C3)C(=O)NC4=CC=CC=C4',
  'formula': 'C33H35FN2O5',
  'molecular_weight_g_mol': 558.6,
  'pubchem_XLogP': 5,
  'pubchem_TPSA_A2': 112,
  'source_url': 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/atorvastatin/property/MolecularFormula,MolecularWeight,CanonicalSMILES,IsomericSMILES,XLogP,TPSA/JSON',
  'source_sha256': 'b2a9fcec871c17fe58c2c0486fb2d6b1400a83868c87f4150d30429c0a9e8dcd',
  'property_kind': 'database representation/computed descriptor; not an assay or measured solubility'},
 {'name': 'atorvastatin_calcium',
  'cid': 60822,
  'smiles': 'CC(C)C1=C(C(=C(N1CC[C@H](C[C@H](CC(=O)[O-])O)O)C2=CC=C(C=C2)F)C3=CC=CC=C3)C(=O)NC4=CC=CC=C4.CC(C)C1=C(C(=C(N1CC[C@H](C[C@H](CC(=O)[O-])O)O)C2=CC=C(C=C2)F)C3=CC=CC=C3)C(=O)NC4=CC=CC=C4.[Ca+2]',
  'formula': 'C66H68CaF2N4O10',
  'molecular_weight_g_mol': 1155.3,
  'pubchem_XLogP': None,
  'pubchem_TPSA_A2': 229,


## Кейс 1. Составить реестр

Для каждой записи сохраните исходную строку, CID/источник, формулу, массу, суммарный заряд и число несвязанных компонентов. Для соли отдельно выпишите формулы и заряды фрагментов. Покажите, где находится анион аторвастатина. Если используете его как четвёртую форму, явно укажите, что выделили его из исходной солевой записи, а не нашли новое экспериментальное измерение.

<details><summary>Подсказка по инструменту</summary>`Chem.GetMolFrags(mol, asMols=True)` возвращает фрагменты как отдельные графы. Сумма их зарядов должна совпадать с зарядом исходной записи.</details>

In [3]:
form_registry=None


## Кейс 2. Проверить предложение коллеги

Коллега хочет взять самый большой фрагмент соли, нейтрализовать его и сохранить под тем же именем, что исходную запись. Объясните, какие сведения исчезнут и для какой задачи такое parent-представление может быть полезно. Затем выясните, является ли лактон просто иной протонированной формой кислоты. Используйте граф/формулу и сохраните карту переходов с условиями, которые ещё предстоит установить.

In [4]:
normalization_decision=None
form_map=None


## Кейс 3. Передать задачу аналитической лаборатории

Рассчитайте число активных органических эквивалентов в выбранной **идеальной безводной записи** кальциевой соли. Объясните, почему этих сведений недостаточно для расчёта реальной навески неизвестного материала. Какие сведения о воде, чистоте и основе количественного содержания вы запросите? Предложите метод подтверждения формы и следующий опыт для проверки возможного превращения кислота/лактон.

In [5]:
stoichiometric_check=None
laboratory_request=None
answer=None
print(submission_feedback(answer))


{'status': 'NOT_RUN', 'scope': 'student work', 'message': 'Сначала выполните собственный кейс и объясните результат.'}


В раздел 2 общего досье добавьте реестр и политику представлений; в раздел 9 — запрос недостающего свидетельства. Приёмка: соль не объявлена одной нейтральной ковалентной молекулой, количество органических компонентов сохранено, лактон не «исправлен» нейтрализацией, реальная гидратация не выдумана.

# M03. Разбор научной статьи: от утверждения к проверке

Основная работа: **Abo Horan et al. (2025). The Chemical Stability Characterization and Kinetics of Statins in Aqueous Cyclodextrin Ocular Preparations: A Formulation Perspective**. [Полный локальный текст](../../../../../shared/data/source-responses/statin_stability_verified.xml), [издательская версия DOI](https://doi.org/10.3390/pharmaceutics17070808). Прочитайте: **Methods и Figure 2 / Table 1**.

## Перенос B: самостоятельная мини-рецензия

Постройте карту форм аторвастатина: исходное вещество → условия → наблюдаемое превращение. Укажите, где заряд и соль требуют отдельной информации.

Извлеките подпись Figure 2 и заголовки Table 1; сохраните таблицу pH, стартовой формы, единицы константы и места в источнике.

Выпишите короткое утверждение своими словами, точный раздел/Table/Figure ID, условия и единицы, способ получения данных и допустимый вывод. Помечайте отдельно опубликованные наблюдения, своё вычисление и гипотезу. Если нужных исходных точек, повторов или параметров нет, запишите «не предоставлено» и объясните, какую проверку это блокирует.

Контрольный вопрос рецензента: **Можно ли перенести константу формы из одной среды на любую запись аторвастатина?** Ответьте со ссылкой на конкретное свидетельство, а не общим «нужно больше исследований».

## Вклад в досье и обсуждение

Сохраните `article_reviews/M03.md` внутри личного `my_research` и численную таблицу/код рядом. Используйте [шаблон мини-рецензии](../../../../../../programm/ARTICLE_REVIEW_TEMPLATE.md). Свяжите запись с вопросом и одним уже принятым решением досье: что статья подтверждает, ограничивает или заставляет изменить? Завершите планом одного независимого опыта с ожидаемым наблюдением, альтернативным исходом и критерием пересмотра.

Оценивание — [единая рубрика](../../../../../../programm/ARTICLE_REVIEW_RUBRIC.md). Ответ обсуждается с преподавателем; запуск Notebook и заполнение шаблона не означают принятия научного аргумента.